# Figure 5 — rare-variant eQTL constraint & sequence-model variant-effect dissection


This notebook reproduces **Figure 5**. The figure pairs a population / constraint
view of fine-mapped **rare** eQTLs at ClinGen disease genes (A–C) with two worked
sequence-model dissections of causal regulatory variants: the **ACVRL1** promoter
(PromoterAI, D/E) and the **SPR** regulatory element (Borzoi allelic browser, F/G).

It runs the modular pipeline in `Fig5/` step by step: a shared **setup** with
memoized loaders (`get_reference()`, `get_eqtl()`, `get_variant_freq()`), one cell
per **panel** (each builds a single *untagged* object), then **assembly**, where
the panel tags A–G are applied. Additional AlphaGenome analyses and supplements
live in `Fig5/extra/` and are not run here.


## How to run

* **Kernel:** R (IRkernel). **Working directory:** the repository root (the folder that contains `Fig5/`).
* Run the cells top to bottom. Each panel caches its ggplot object to `Fig5/objects/`, and the assembly cell reads those back.
* Re-running a single panel cell rebuilds just that panel; re-run the assembly cell to recompose.


### Inputs

> **Controlled-access.** The SuSiE eQTL table and the no-homozygote list derive
> from *All of Us* controlled-access data and are not in this repository; the SPR
> Borzoi bigWigs are large model-derived files. Public / derived reference tables
> (ClinGen, PromoterAI, ENCODE cCREs, GENCODE gene models) are committed under
> `Fig5/reference/`.

Inputs are **manifest-driven** (`Fig5/inputs.tsv`). Panel C degrades gracefully
when the private no-homozygote list is absent.

> **Panel B/C note.** A gene curated under **both** AD and AR ClinGen inheritance
> is counted under both classes (preserved from the original). See `Fig5/NOTES.md`.


## Setup

`00_setup.R` is the shared foundation every step builds on: it loads the libraries, sets the **Inter** font with a matched output **DPI**, defines the `theme_set`/`SZ_*` design system and the env-overridable paths, and brings in the manifest loaders and the folded-in analysis / plot functions — so the figure is self-contained.


In [ ]:
## Locate the figure directory. Terra / Linux is case-sensitive, so match
## either "Fig5" or "fig5" and point the pipeline's env-overridable
## paths at it before sourcing the shared setup.
fig_dir <- list.files(".", pattern = "^(Fig5|fig5)$", include.dirs = TRUE)
stopifnot("Run this notebook from the repository root (the folder holding Fig5/)." =
          length(fig_dir) == 1)
Sys.setenv(FIG5_MANIFEST = file.path(fig_dir, "inputs.tsv"),
           FIG5_OBJ = file.path(fig_dir, "objects"),
           FIG5_DATA = file.path(fig_dir, "data"))
source(file.path(fig_dir, "00_setup.R"))

## Inline rendering. The canvas resolution is matched to DPI because 00_setup.R
## pins showtext to it (showtext_opts(dpi = DPI)); if the two disagree, every
## label renders at the wrong size. Panels display at a readable default, and
## the assemble cell overrides these with the composed figure's true aspect.
options(repr.plot.width = 7.5, repr.plot.height = 5, repr.plot.res = DPI)
cat("Setup loaded from", fig_dir, "\n")


### Fig5 Panel A — fine-mapped rare ClinGen variants by ancestry (AD/AR) over a per-inheritance ClinVar-classification proportion bar. Untagged; run from root.

Fine-mapped rare ClinGen variants by genetic ancestry, faceted AD vs AR (lighter = novel, darker = ClinVar-annotated), over a per-inheritance ClinVar-classification proportion bar. A 2-plot patchwork; tagged A.


In [ ]:
ref  <- get_reference()
eqtl <- get_eqtl()

A <- build_panelA(eqtl, ref$clingen)
saveRDS(A, file.path(OBJ_DIR, "A.rds"))
message("Panel A built.")

A


### Fig5 Panel B — |average QTL effect| by variant-effect category (Control/AD/AR), with 95% CIs. Untagged; run from the project root.

|Average QTL effect| by variant-effect category (Control / AD / AR, split Aggregate / Decreasing / Increasing) with 95% CIs. Plots |posterior_mean| (the SuSiE effect), not log2 aFC. A gene curated under BOTH AD and AR is counted in both classes (preserved fan-out; see Fig5/NOTES.md). Tagged B.


In [ ]:
ref  <- get_reference()
eqtl <- get_eqtl()

B <- build_panelB(eqtl, ref$clingen)
saveRDS(B, file.path(OBJ_DIR, "B.rds"))
message("Panel B built.")

B


### Fig5 Panel C — log2 aFC of the strongest decreasing variant per gene (top 7 per AD/AR). Uses the optional private no-homozygote list if present. Untagged.

log2 aFC of the strongest decreasing variant per gene, top 7 per AD/AR (HEXA and MLC1 forced in), optionally refined by the private no-homozygote list. Tagged C.


In [ ]:
ref  <- get_reference()
eqtl <- get_eqtl()

C <- build_panelC(eqtl, ref$clingen, nohom = load_nohomozygotes())
saveRDS(C, file.path(OBJ_DIR, "C.rds"))
message("Panel C built.")

C


### Fig5 Panel D — ACVRL1 eQTL fine-mapping (regional PIP + zoom inset + gene track). Untagged; run from the project root.

ACVRL1 eQTL fine-mapping: regional PIP with a zoom inset and an aligned gene-model track, ENCODE cCREs shaded. A patchwork wrapped as one tag leaf; tagged D.


In [ ]:
eqtl <- get_eqtl()

D <- build_panelD(eqtl)
saveRDS(D, file.path(OBJ_DIR, "D.rds"))
message("Panel D built.")

D


### Fig5 Panel E — ACVRL1 causal promoter variants, PromoterAI base-resolution effect (signed-square y-axis). Untagged; run from the project root.

ACVRL1 causal promoter variants — PromoterAI base-resolution effect on a signed-square axis, one window per causal variant; the focal chr12:51,912,316 C>T sits in an SP1 GC-box. Tagged E.


In [ ]:
eqtl <- get_eqtl()
vf   <- get_variant_freq()

E <- build_panelE(eqtl, vf)
saveRDS(E, file.path(OBJ_DIR, "E.rds"))
message("Panel E built.")

E


### Fig5 Panel F — SPR eQTL fine-mapping (regional PIP + zoom inset + gene track). Untagged; run from the project root.

SPR eQTL fine-mapping: regional PIP + zoom inset + gene-model track. A patchwork wrapped as one tag leaf; tagged F.


In [ ]:
eqtl <- get_eqtl()

F_panel <- build_panelF(eqtl)
saveRDS(F_panel, file.path(OBJ_DIR, "F.rds"))
message("Panel F built.")

F_panel


### Fig5 Panel G — SPR Borzoi multi-track allelic browser + CTCF attribution logo. A cowplot object wrapped for patchwork. Untagged; run from the project root.

SPR Borzoi multi-track allelic browser (reference vs alternate across 16 marks, broad→narrow zoom) with the CTCF base-resolution attribution logo. A cowplot object; tagged G.


In [ ]:
eqtl <- get_eqtl()
vf   <- get_variant_freq()

G <- build_panelG(eqtl, vf)
saveRDS(G, file.path(OBJ_DIR, "G.rds"))
message("Panel G built.")

G


## Assemble the figure

Loads the cached panel objects from `Fig5/objects/` and composes them with patchwork into the final figure. Panel **tagging happens here and only here**. Writes the PDF plus a PNG preview, and renders the assembled figure inline.


In [ ]:
P <- setNames(lapply(LETTERS[1:7],
                     function(x) readRDS(file.path(OBJ_DIR, paste0(x, ".rds")))),
              LETTERS[1:7])

# Collapse the composite panels into single tag leaves (G is already wrapped).
A_panel <- wrap_elements(full = P$A)
D_panel <- wrap_elements(full = P$D)
F_panel <- wrap_elements(full = P$F)

combined_BC <- (P$B | P$C) + plot_layout(widths = c(0.85, 1.05))
top_row     <- A_panel | combined_BC + plot_layout(widths = c(1.5, 1))

finemap_row   <- D_panel | F_panel
mechanism_row <- P$E | P$G
bottom_half   <- finemap_row / plot_spacer() / mechanism_row +
  plot_layout(heights = c(1, 0.10, 2.33))

Fig5 <- top_row / plot_spacer() / bottom_half +
  plot_layout(heights = c(1, 0.08, 2.43)) +
  plot_annotation(tag_levels = list(c("A", "B", "C", "D", "F", "E", "G"))) &
  theme(plot.tag = element_text(size = SZ_TAG, face = "bold", family = FONT))

ggsave(file.path(OUTDIR, "Figure5.pdf"), Fig5, width = 18, height = 16, bg = "white",
       dpi = DPI, limitsize = FALSE)
showtext_opts(dpi = 100)
ggsave(file.path(OUTDIR, "Figure5.png"), Fig5, width = 18, height = 16, bg = "white",
       dpi = 100, limitsize = FALSE)
showtext_opts(dpi = DPI)

message("Assembled: ", file.path(OUTDIR, "Figure5.pdf"), " and Figure5.png")

options(repr.plot.width = 12.0, repr.plot.height = 10.7)
Fig5
